# Extended SIR model for follicle life cycle
The following graph is a state machine representation of the life cycle of a follicle  

```mermaid
graph TD
A((A))
T((T))
C((C))

A --> C
C --> T
T --> A

%% Styling to shrink nodes
style A width:20px,height:20px,font-size:10px
style T width:20px,height:20px,font-size:10px
style C width:20px,height:20px,font-size:10px
```

Where:
- A Anagen is a growing state
- C Catagen is a transition stage
- T Talogen is a resting state

We used SIR-like modeling

$$
\frac{dA}{dt} = k_{wake}(\frac{w_{nt}^n}{w_{nt}^n + k_{wnt}^n})T - A(\beta_{base} + \beta_{max}(\frac{FGF5^n}{FGF5^n + K_{FGF5}^n} .\frac{DHT^m}{DHT^m + K_{DHT}^m} ))
$$
$$
\frac{dC}{dt} = A(\beta_{base} + \beta_{max}(\frac{FGF5^n}{FGF5^n + K_{FGF5}^n} .\frac{DHT^m}{DHT^m + K_{DHT}^m} )) - \gamma C
$$
$$
\frac{dT}{dt} =  \gamma C - k_{wake}(\frac{w_{nt}^n}{w_{nt}^n + k_{wnt}^n})T
$$
where:
- $k_{wake}$ is the maximum fraction of T that converts to A at a time
- $w_{nt}$ is a protein signaling awakening of follicles (converting from Talogen to Anagen)
- FGF5 is a protein that signals the transition from Anagen to Catagen. It is a growth inhibitor.
- DHT is a hormone that accelerates the FGF5 effect.
- $\gamma$ is the C to T conversion rate. It solely depends on the biological clock, not on hormones or proteins.

Treatments like minoxidal lowers the $k_{wnt}$ and Finosteride reduces DHT

Parameter variations:

- $K_{FGF5}$ and $K_{DHT}$ vary from person to person as they have strong genetic dependency
- DHT varies depending on age and health condition. More DHT at older ages.
- FGF5 and $W_{nt}$ can be assumed among people as they are produced at the same rate and quantity.
- $K_{wnt}$ also depends on genetics but can be assumed constant for people of similar age and health condition.


# Nonlinear regression to determine the parameters in the extended-SIR model.

There are no publicly available clinical datasets on hair loss or hair treatment that are directly relevant to this project. In particular, longitudinal measurements—such as follicle counts in the anagen (A) and telogen (T) phases, average follicle diameter, and related metrics—would be highly valuable. Such data could be used in a regression framework to estimate the parameters of an SIR-type model describing follicle state dynamics.

In the absence of suitable public datasets, we relied on Halloy et al. to generate synthetic data. Their work models the transition times between follicle states using log-normal distributions parameterized by experimentally derived values. Building on this framework, we conducted an agent-based simulation to produce longitudinal data, specifically tracking the number of follicles in the anagen (A), catagen (C), and telogen (T) states over time. We then fit the extended SIR model to the mean trajectories obtained by averaging across multiple simulation runs. 

The distribution of the durations of each follicular phase is represented by a log-normal function:

$$
f(x;\,\mu,\sigma)
= \frac{1}{x\,\sigma\sqrt{2\pi}}
\exp\!\left(
-\frac{(\log x - \mu)^2}{2\sigma^2}
\right),
\qquad x>0 \tag{1}
$$

The mean and variance of the duration are determined from the experimental data. Using the moment generating function, we can calculate the parameters of the normal for the log-normal distribution as:
$$
\sigma _{\log }^2=\ln \! \left( 1+\frac{v}{m^2}\right) \tag{3a}
$$

$$
\mu _{\log }=\ln (m)-\frac{1}{2}\sigma _{\log }^2 \tag{3b}
$$



# I. Generate Synthetic Data

In [34]:
#---------------------------------------------------------
# 1. Log-normal duration sampler
#    (Phase durations drawn from empirical distributions)
#---------------------------------------------------------
def sample_duration(mu, sigma):
    """Sample a phase duration from the log-normal distribution in Eq. (1)."""
    return np.random.lognormal(mean=mu, sigma=sigma)


#---------------------------------------------------------
# 2. Probability of transition into pathological state M
#    (Slowly increasing probability as cycles accumulate)
#---------------------------------------------------------
def p_to_M(cycles):
    """Slowly increasing probability of transitioning to M."""
    return min(0.01 * (1 + 0.1 * cycles), 0.2)

# def p_to_M_time(t, T_end):
#     base = 0.002
#     ramp = 0.01 * (t / T_end)
#     return min(base + ramp, 0.2)


#---------------------------------------------------------
# 3. Baseline follicle-cycle structure (A → T → L → A)
#---------------------------------------------------------
next_phase = {
    "A": "T",
    "T": "L",
    "L": "A"
}


#-----------------------------------------------------------
# 5. Phase-duration parameters for A, T, L, and optional M
#-----------------------------------------------------------
def convert_to_lognormal_params(mean, variance):
    sigma2_log = np.log(1 + variance / (mean**2))
    sigma_log = np.sqrt(sigma2_log)
    mu_log = np.log(mean) - 0.5 * sigma2_log
    return mu_log, sigma_log

# phase_params = {
#     "A": convert_to_lognormal_params(6.49, 10.16),
#     "T": convert_to_lognormal_params(2.17, 1.11),
#     "L": convert_to_lognormal_params(4.56, 9.57)
# }

mu_M = 3.0      # example mean duration of M
var_M = 2.0     # example variance of M

phase_params = {
    "A": convert_to_lognormal_params(6.49, 10.16),
    "T": convert_to_lognormal_params(2.17, 1.11),
    "L": convert_to_lognormal_params(4.56, 9.57),
    "M": convert_to_lognormal_params(mu_M, var_M)   # you choose these
}



# phase_params = {
#     "A": convert_to_lognormal_params(16.91, 19.49**2),
#     "T": convert_to_lognormal_params(1.79, 0.82**2),
#     "L": convert_to_lognormal_params(5.23, 5.18**2)
# }


#---------------------------------------------------------
# 6. Phase-transition rule
#    (Normal cycling or optional transition into M)
#---------------------------------------------------------
def choose_next_phase(phase, cycles, include_M=True):
    """
    Determine the next phase in the follicle cycle.
    If include_M=True, allow a slow-probability transition to M.
    """
    # If follicle is already in M, it stays there
    if phase == "M":
        return "M"

    # Normal cycle transitions
    next_phase_map = {"A": "T", "T": "L", "L": "A"}

    # If M transitions are disabled
    if not include_M:
        return next_phase_map[phase]

    # Otherwise include slow probability to M
    pM = p_to_M(cycles)
    # pM = p_to_M_time(t, T_end)

    if np.random.rand() < pM:
        return "M"
    else:
        return next_phase_map[phase]


#---------------------------------------------------------
# 7. Simulate a single follicle over T_end months
#---------------------------------------------------------
def simulate_follicle(T_end, initial_phase="A", include_M=True):
    phase = initial_phase
    cycles = 0
    time_to_transition = sample_duration(*phase_params[phase])

    phases = []
    cycle_counts = []

    for t in range(T_end):

        # If already in M, stay in M
        if phase == "M":
            phases.append("M")
            cycle_counts.append(cycles)
            continue

        phases.append(phase)
        cycle_counts.append(cycles)

        time_to_transition -= 1

        if time_to_transition <= 0:

            # Use the new choice function
            phase = choose_next_phase(phase, cycles, include_M=include_M)

            # If moved to M, no new duration needed
            if phase == "M":
                continue

            # Otherwise sample new duration
            time_to_transition = sample_duration(*phase_params[phase])

            # Count cycles when entering A
            if phase == "A":
                cycles += 1

    return phases, cycle_counts


#---------------------------------------------------------
# 8. Simulate a population of independent follicles
#---------------------------------------------------------
def simulate_population(N_follicles, T_end, include_M=True):
    all_phases = []
    all_cycles = []

    for i in range(N_follicles):
        phases, cycles = simulate_follicle(T_end, include_M=include_M)
        all_phases.append(phases)
        all_cycles.append(cycles)

    return np.array(all_phases), np.array(all_cycles)


#---------------------------------------------------------
# 9. Population-level statistics at a given time t
#---------------------------------------------------------
def population_statistics(all_phases, all_cycles, t=None):
    """
    Compute number of follicles in each phase at time t,
    and mean number of cycles performed.
    """
    if  t:
        phases_t = all_phases[:, t]
        cycles_t = all_cycles[:, t]
    
        return {
            "A": np.sum(phases_t == "A"),
            "C": np.sum(phases_t == "T"),
            "T": np.sum(phases_t == "L"),
            #"M": np.sum(phases_t == "M"),
            "mean_cycles": np.mean(cycles_t)
        }
    else:
        return {
            "A": np.sum(all_phases == "A", axis=0),
            "C": np.sum(all_phases == "T", axis=0),
            "T": np.sum(all_phases == "L", axis=0),
            #"M": np.sum(phases_t == "M"),
            "mean_cycles": np.mean(all_cycles, axis=0)
        }




In [35]:
def simulate_follicle2(T_end, initial_phase="A", include_M=False):
    phase = initial_phase
    cycles = 0
    time_to_transition = sample_duration(*phase_params[phase])
    remaining = T_end
    moves=0
    times =[]
    while remaining >0:
        time_to_transition = sample_duration(*phase_params[phase])
        times.append(time_to_transition)
        remaining -= time_to_transition
        if remaining >0:
            moves+=1
            phase = next_phase[phase]
    full_cycles =  moves//3
    partial_cycle = moves %3
    phases = list('ACT'*full_cycles + 'ACT'[:partial_cycle+1]) 
    cycle_counts = full_cycles

    return phases, cycle_counts, times

In [38]:
#---------------------------------------------------------
# 10. Paired simulations: with-M vs without-M transitions
#---------------------------------------------------------
def run_two_simulations(N, T_end):
    # With M transitions
    phases_M, cycles_M = simulate_population(N, T_end, include_M=True)

    # Without M transitions
    phases_noM, cycles_noM = simulate_population(N, T_end, include_M=False)

    return (phases_M, cycles_M), (phases_noM, cycles_noM)






In [45]:
#---------------------------------------------------------
# 11. Generate data
#---------------------------------------------------------
N = 1000       # follicles
T_end = 200    # months
n_simulation = 25

A, C, T = [],[],[]
for i in range(n_simulation):
    all_phases, all_cycles = simulate_population(N, T_end)
    data= population_statistics(all_phases, all_cycles)
    A.append(data['A'])
    C.append(data['C'])
    T.append(data['T'])


# II. Apply Nonlinear Regression

# III Parameter Sensitivity and Hair Treatment Analysis